# Feature Engineering: Inspection Outcome Prediction Pipeline

Binary classification target: **Passed** vs **Needs Action**.
Source spec: `docs/feature_engineering_spec.md`. Output: `data/feature_matrix.csv`.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

BASE = Path('../data')

## 1. Data Loading

In [2]:
df_inspection = pd.read_csv(
    BASE / 'inspection.csv',
    parse_dates=['Earliest_INSPECTION_Date', 'Latest_INSPECTION_Date'],
)
df_order = pd.read_csv(BASE / 'order.csv')
df_order['DateofIssue'] = pd.to_datetime(df_order['DateofIssue'], errors='coerce')
df_static = pd.read_csv(BASE / 'merged_elevator_data.csv')

df_inspection = df_inspection.rename(columns={'Latest_INSPECTION_Date': 'inspection_date'})

print(f'inspection : {df_inspection.shape}')
print(f'order      : {df_order.shape}')
print(f'static     : {df_static.shape}')
print(f'\nDateofIssue dtype: {df_order["DateofIssue"].dtype}')
print(f'inspection_date dtype: {df_inspection["inspection_date"].dtype}')

inspection : (143181, 9)
order      : (162172, 15)
static     : (52339, 34)

DateofIssue dtype: datetime64[us]
inspection_date dtype: datetime64[us]


## 2. Clean `merged_elevator_data.csv`

Keep only the four columns needed by the pipeline. Replace redacted placeholder strings with `NaN` before selecting — `location` contains address data that could be redacted.

In [3]:
df_static = df_static.replace({'data redacted': np.nan, 'redacted': np.nan})

df_static = df_static.rename(columns={
    'LocationoftheElevatingDevice': 'location',
    'Device Type': 'equipment_type',
    'Device Class': 'device_class',
})
df_static = df_static[['ElevatingDevicesNumber', 'equipment_type', 'device_class', 'location']]

print(f'Static cleaned: {df_static.shape}')
print(f'Null counts:\n{df_static.isnull().sum()}')

Static cleaned: (52339, 4)
Null counts:
ElevatingDevicesNumber    0
equipment_type            0
device_class              0
location                  0
dtype: int64


## 3. Clean Inspection Outcome (Target Variable)

In [4]:
EXCLUDE_OUTCOMES = {
    'Unable to Inspect', 'Incomplete', 'Not Required', 'Dismantled',
    'Cancelled', 'Extend Time to Comply', 'Undergoing Major Alt',
    'Closed by Program', 'RC Established', 'Temp Lic Not Needed',
    'Received', 'Order Transferred',
}

PASS_OUTCOMES = {
    'Passed', 'Passed Major', 'Passed Sub', 'All Orders Resolved',
    'Complete', 'Inspection Complete', 'Complete Enforcement',
}

n_before = len(df_inspection)
df_inspection = df_inspection[~df_inspection['InspectionOutcome'].isin(EXCLUDE_OUTCOMES)].copy()
n_excluded = n_before - len(df_inspection)
print(f'Rows excluded (ambiguous/administrative): {n_excluded:,}')

def map_outcome(val: str) -> str | None:
    v = str(val).strip()
    if v in PASS_OUTCOMES:
        return 'Passed'
    if re.search(r'follow.{0,5}up', v, re.IGNORECASE):
        return 'Needs Action'
    if v in {'Shutdown', 'Vol Shut Down', 'Fail', 'Fail Initial', 'Fail Sub'}:
        return 'Needs Action'
    return None

df_inspection['outcome_binary'] = df_inspection['InspectionOutcome'].map(map_outcome)

unmapped = df_inspection['outcome_binary'].isna()
if unmapped.any():
    print(f'Unmapped outcomes ({unmapped.sum()} rows):')
    print(df_inspection.loc[unmapped, 'InspectionOutcome'].value_counts())
df_inspection = df_inspection.dropna(subset=['outcome_binary']).copy()

dist = df_inspection['outcome_binary'].value_counts()
majority = dist.max() / dist.sum()
print(f'\nClass distribution:\n{dist}')
print(f'\nMajority-class baseline accuracy: {majority:.3f} ({majority:.1%})')

Rows excluded (ambiguous/administrative): 1,301

Class distribution:
outcome_binary
Needs Action    87935
Passed          53945
Name: count, dtype: int64

Majority-class baseline accuracy: 0.620 (62.0%)


## 4. Clean Inspection Type (Feature)

In [5]:
# Fix double-space typo
df_inspection['InspectionType'] = df_inspection['InspectionType'].str.replace(
    'ED-Sub  Inspection', 'ED-Sub Inspection', regex=False
)

TYPE_MAP = {
    'ED-Periodic Inspection': 'Periodic',
    'ED-Followup Inspection': 'Followup',
    'ED-Followup Minor Alt': 'Followup',
    'ED-Followup Ownership Change': 'Followup',
    'ED-MCP Follow up': 'Followup',
    'ED-FU Enforcement Action Insp': 'Followup',
    'ED-Followup Lic Insp': 'Followup',
    'ED-Followup No-Lic Insp': 'Followup',
    'ED-Followup Reg Non-Compliance': 'Followup',
    'ED-Non-Mandated Followup ON': 'Followup',
    'ED-PWGSC Foll-Up': 'Followup',
    'ED-Minor A Inspection': 'Alteration',
    'ED-Minor B Inspection': 'Alteration',
    'ED-Sub Inspection Major': 'Alteration',
    'ED-Major Alteration Inspection': 'Alteration',
    'ED-Sub Inspection': 'Sub',
    'ED-Sub Failed Initial': 'Sub',
    'ED-Initial Inspection': 'Initial',
    'ED-Unscheduled Inspection': 'Unscheduled',
    'ED-Enforcement Action': 'Enforcement',
    'ED-MCP Enforcement Insp': 'Enforcement',
}

n_before = len(df_inspection)
df_inspection['inspection_type_cleaned'] = df_inspection['InspectionType'].map(TYPE_MAP)

unmapped_types = df_inspection['inspection_type_cleaned'].isna()
if unmapped_types.any():
    print(f'Unmapped types ({unmapped_types.sum()} rows):')
    print(df_inspection.loc[unmapped_types, 'InspectionType'].value_counts())

df_inspection = df_inspection.dropna(subset=['inspection_type_cleaned']).copy()
print(f'Rows dropped (unmapped type): {n_before - len(df_inspection):,}')
print(f'Remaining: {len(df_inspection):,}')

df_cleaned = df_inspection.copy()

Unmapped types (91 rows):
InspectionType
ED-Re-Activate Inspection    74
ED-Non-Mandated Insp ON       6
ED-PWGSC Insp                 5
ED-Inspection Temp Lic        5
ED-Reg Non-Compliance         1
Name: count, dtype: int64
Rows dropped (unmapped type): 91
Remaining: 141,789


## 5. Temporal Features from Prior Inspections

For each inspection row, features are computed from all prior inspections of the same device
where `inspection_date < current_inspection_date` (strict). Elevators with no prior history
receive count = 0 and NaN for date/outcome features.

**Rolling pass rate** uses the last 5 prior inspections — recent history is most predictive,
and a window of 5 balances signal quality with coverage for devices with few records.

In [6]:
TYPE_GROUPS = ['Periodic', 'Followup', 'Alteration', 'Sub', 'Initial', 'Unscheduled', 'Enforcement']

def compute_temporal_features(grp: pd.DataFrame) -> pd.DataFrame:
    grp = grp.sort_values('inspection_date')
    n = len(grp)
    dates = grp['inspection_date'].values
    outcomes = grp['outcome_binary'].values
    types = grp['inspection_type_cleaned'].values

    records = []
    for i in range(n):
        mask = dates < dates[i]
        prior_idx = np.where(mask)[0]
        cnt = len(prior_idx)

        rec: dict = {
            'prior_inspection_count': cnt,
            'prior_outcome_counts_passed': 0,
            'prior_outcome_counts_needs_action': 0,
            'days_since_last_inspection': np.nan,
            'rolling_pass_rate': np.nan,
            'most_recent_prior_outcome': np.nan,
        }
        for tg in TYPE_GROUPS:
            rec[f'prior_type_counts_{tg.lower()}'] = 0

        if cnt > 0:
            prior_outcomes = outcomes[prior_idx]
            rec['prior_outcome_counts_passed'] = int((prior_outcomes == 'Passed').sum())
            rec['prior_outcome_counts_needs_action'] = int((prior_outcomes == 'Needs Action').sum())

            prior_types = types[prior_idx]
            for tg in TYPE_GROUPS:
                rec[f'prior_type_counts_{tg.lower()}'] = int((prior_types == tg).sum())

            last_idx = prior_idx[-1]
            delta = dates[i] - dates[last_idx]
            rec['days_since_last_inspection'] = float(delta / np.timedelta64(1, 'D'))
            rec['most_recent_prior_outcome'] = outcomes[last_idx]

            window_idx = prior_idx[-5:]
            rec['rolling_pass_rate'] = float((outcomes[window_idx] == 'Passed').mean())

        records.append(rec)

    return pd.DataFrame(records, index=grp.index)

df_cleaned = df_cleaned.sort_values(['ElevatingDevicesNumber', 'inspection_date']).reset_index(drop=True)

print('Computing temporal features — may take ~1 min...')
temporal_df = df_cleaned.groupby('ElevatingDevicesNumber', group_keys=False).apply(compute_temporal_features)
df_feat = df_cleaned.join(temporal_df)

first_count = (df_feat['prior_inspection_count'] == 0).sum()
print(f'First-ever inspections (no prior history): {first_count:,}')
print(f'Shape after temporal features: {df_feat.shape}')

Computing temporal features — may take ~1 min...
First-ever inspections (no prior history): 43,282
Shape after temporal features: (141789, 24)


## 6. Prior Order Features

Orders are joined strictly by `DateofIssue < inspection_date` (date-level comparison).
RISKSCORE nulls (~25%) are excluded from the mean — imputing with 0 would imply no risk
for records that simply predate the scoring system.

In [7]:
n_orders = len(df_order)
n_missing_risk = df_order['RISKSCORE'].isna().sum()
print(f'RISKSCORE missing: {n_missing_risk:,} / {n_orders:,} ({n_missing_risk/n_orders:.1%})')
print('Strategy: mean over non-null values; NaN if no non-null prior orders.\n')

df_order = df_order.copy()
df_order['order_date'] = df_order['DateofIssue'].dt.normalize()
df_order_sorted = df_order.sort_values(['ElevatingDevicesNumber', 'order_date'])

order_lookup: dict = {}
for device, grp in df_order_sorted.groupby('ElevatingDevicesNumber'):
    order_lookup[device] = {
        'dates': pd.DatetimeIndex(grp['order_date'].values),
        'risks': grp['RISKSCORE'].values.astype(float),
    }

def _order_features_for_device(device, grp: pd.DataFrame) -> pd.DataFrame:
    counts = []
    avg_risks = []

    if device not in order_lookup:
        return pd.DataFrame(
            {'prior_order_count': [0] * len(grp), 'prior_avg_risk_score': [np.nan] * len(grp)},
            index=grp.index,
        )

    dates = order_lookup[device]['dates']
    risks = order_lookup[device]['risks']

    for t in grp['inspection_date']:
        t_norm = t.normalize()
        pos = int(dates.searchsorted(t_norm, side='left'))
        counts.append(pos)
        if pos > 0:
            valid = risks[:pos][~np.isnan(risks[:pos])]
            avg_risks.append(float(valid.mean()) if len(valid) > 0 else np.nan)
        else:
            avg_risks.append(np.nan)

    return pd.DataFrame(
        {'prior_order_count': counts, 'prior_avg_risk_score': avg_risks},
        index=grp.index,
    )

print('Computing prior order features...')
chunks = []
for device, grp in df_feat.groupby('ElevatingDevicesNumber'):
    chunks.append(_order_features_for_device(device, grp))
order_features = pd.concat(chunks)
df_feat = df_feat.join(order_features)
print(f'Shape after order features: {df_feat.shape}')

RISKSCORE missing: 41,553 / 162,172 (25.6%)
Strategy: mean over non-null values; NaN if no non-null prior orders.

Computing prior order features...
Shape after order features: (141789, 26)


## 7. Join Static Features

In [8]:
# merged_elevator_data has multiple rows per device (one per alteration record)
# Deduplicate to one row per device before joining
df_static_unique = df_static.drop_duplicates(subset=['ElevatingDevicesNumber'], keep='first')
print(f'Unique devices in static: {len(df_static_unique):,}')

df_feat = df_feat.merge(
    df_static_unique,
    on='ElevatingDevicesNumber',
    how='left',
)

unmatched = df_feat['equipment_type'].isna().sum()
print(f'Shape after join: {df_feat.shape}')
print(f'Inspections with no static record: {unmatched:,} ({unmatched/len(df_feat):.1%})')

Unique devices in static: 43,154
Shape after join: (141789, 29)
Inspections with no static record: 2,994 (2.1%)


## 8. Encode Dummy Variables

Sentinel values are filled before encoding so all categories are represented.
- `days_since_last_inspection` NaN → `-1` (no prior history)
- `rolling_pass_rate` NaN → `0.0` (no prior history)
- `most_recent_prior_outcome` NaN → `'NO_HISTORY'`
- `prior_avg_risk_score` NaN → `0.0` (no prior orders or no scored orders)
- Static columns NaN → `'Unknown'` (device not in static table)

In [9]:
df_feat['days_since_last_inspection'] = df_feat['days_since_last_inspection'].fillna(-1)
df_feat['rolling_pass_rate'] = df_feat['rolling_pass_rate'].fillna(0.0)
df_feat['most_recent_prior_outcome'] = df_feat['most_recent_prior_outcome'].fillna('NO_HISTORY')
df_feat['prior_avg_risk_score'] = df_feat['prior_avg_risk_score'].fillna(0.0)
for col in ['equipment_type', 'device_class', 'location']:
    df_feat[col] = df_feat[col].fillna('Unknown')

# Sanitize string values to avoid spaces in column names after encoding
for col in ['most_recent_prior_outcome', 'equipment_type', 'device_class', 'location']:
    df_feat[col] = df_feat[col].str.replace(r'[\s/\-]+', '_', regex=True)

ENCODE_COLS = ['inspection_type_cleaned', 'equipment_type', 'device_class', 'most_recent_prior_outcome']
df_feat = pd.get_dummies(df_feat, columns=ENCODE_COLS, drop_first=True)
df_feat.columns = [c.replace(' ', '_') for c in df_feat.columns]

print(f'Shape after encoding: {df_feat.shape}')

Shape after encoding: (141789, 41)


## 9. Final Validation

In [10]:
print('=== Leakage Verification ===\n')

# Rows with prior inspections must have days_since > 0 (prior date was strictly earlier)
has_prior = df_feat['prior_inspection_count'] > 0
violations = (df_feat.loc[has_prior, 'days_since_last_inspection'] <= 0).sum()
assert violations == 0, f'Temporal leakage: {violations} violations'
print(f'Temporal leakage check passed — {has_prior.sum():,} rows with prior history, 0 violations')

assert (df_feat['prior_order_count'] >= 0).all(), 'Negative order counts'
print('Order count sanity check passed')

print('\nSample of 20 rows (manual date verification):')
sample_cols = ['ElevatingDevicesNumber', 'inspection_date', 'prior_inspection_count',
               'days_since_last_inspection', 'prior_order_count']
print(df_feat.sample(20, random_state=42)[sample_cols].sort_values('inspection_date').to_string())

print('\n=== Missing Values Check ===\n')

# Columns not in the final matrix are excluded from the null check
_RAW_COLS = {
    'inspection_date', 'originatingservicerequestnumber', 'InspectionCustomer',
    'InspectionLocation', 'InspectionOutcome', 'InspectionType', 'Earliest_INSPECTION_Date',
}
check_cols = [c for c in df_feat.columns if c not in _RAW_COLS]
null_counts = df_feat[check_cols].isnull().sum()
if null_counts.sum() > 0:
    print('Columns with nulls:')
    print(null_counts[null_counts > 0])
else:
    print('Zero nulls across all feature columns')
assert null_counts.sum() == 0, 'Null check failed'

print('\n=== Final Matrix Summary ===')
print(f'Shape: {df_feat.shape}')
print(f'\nClass distribution:')
print(df_feat['outcome_binary'].value_counts())

=== Leakage Verification ===

Temporal leakage check passed — 98,507 rows with prior history, 0 violations
Order count sanity check passed

Sample of 20 rows (manual date verification):
        ElevatingDevicesNumber inspection_date  prior_inspection_count  days_since_last_inspection  prior_order_count
19916                    16977      2011-03-11                       1                         8.0                  0
71825                    39628      2011-04-08                       0                        -1.0                  0
72030                    39701      2011-11-08                       0                        -1.0                  0
92411                    68973      2011-12-12                       0                        -1.0                  0
79130                    62311      2012-01-11                       0                        -1.0                  1
94981                    70696      2012-01-23                       0                        -1.0        

## 10. Output

In [11]:
ID_COLS = ['ElevatingDevicesNumber', 'InspectionNumber', 'inspection_date']
TARGET = 'outcome_binary'

RAW_EXCLUDE = {
    'originatingservicerequestnumber', 'InspectionCustomer', 'InspectionLocation',
    'InspectionOutcome', 'InspectionType', 'Earliest_INSPECTION_Date',
}

feature_cols = [
    c for c in df_feat.columns
    if c not in set(ID_COLS) | {TARGET} | RAW_EXCLUDE
]

df_final = df_feat[ID_COLS + [TARGET] + feature_cols]

output_path = BASE / 'feature_matrix.csv'
df_final.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Final shape: {df_final.shape}')
print(f'ID/date columns (not model inputs): {ID_COLS}')
print(f'Target column: {TARGET}')
print(f'Feature columns: {len(feature_cols)}')

Saved: ../data/feature_matrix.csv
Final shape: (141789, 35)
ID/date columns (not model inputs): ['ElevatingDevicesNumber', 'InspectionNumber', 'inspection_date']
Target column: outcome_binary
Feature columns: 31
